In [7]:
import sys, json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import torch
from rdkit import Chem
from rdkit.Chem import Draw

def find_project_root(marker=".git") -> Path:
    path = Path.cwd()
    for parent in [path, *path.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find project root (no {marker} found)")

project_root = find_project_root()
sys.path.append(str(project_root))

from src.sample import generate_molecules
from src.evaluate import (
    compute_validity, compute_uniqueness, compute_novelty,
    compute_qed_scores, compute_scaffold_novelty,
)

data_dir = project_root / "data" / "processed" / "char_tokenized"
results_dir = project_root / "results"
results_dir.mkdir(parents=True, exist_ok=True)

with open(data_dir / "canonical_training_smiles.json") as f:
    canonical_training_set = set(json.load(f))

with open(data_dir / "holdout_scaffolds.json") as f:
    holdout_scaffolds = set(json.load(f))

df = pd.read_csv(project_root / "data" / "zinc250k.csv")

model_name = "char_lstm_0804_160242"
model_path = project_root / "models" / f"{model_name}.th"

In [8]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA")
elif torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
    print("Using MPS")
else:
    print("CUDA not available, using CPU")
    device = torch.device("cpu")

Using CUDA


In [9]:
temperatures = [0.7, 1.0, 1.3]
results = []
generated_by_temp = {}  # keep the raw output around for the molecule-rendering step later


for temp in temperatures:
    generated = generate_molecules(
        num_samples=2000, model_path=model_path,
        batch_size=256, device=device, temperature=temp,
    )
    generated_by_temp[temp] = generated

    valid_smiles, invalid_count = compute_validity(generated)
    validity_rate = len(valid_smiles) / len(generated)
    uniqueness_rate = compute_uniqueness(valid_smiles)  # duplicates intact -- per the earlier discussion

    unique_valid_smiles = list(set(valid_smiles))
    if unique_valid_smiles:
        novel, not_novel, novelty_rate = compute_novelty(canonical_training_set, unique_valid_smiles)
        scaffold_novelty_rate = compute_scaffold_novelty(unique_valid_smiles, holdout_scaffolds)
    else:
        novelty_rate = scaffold_novelty_rate = 0.0

    results.append({
        "temperature": temp,
        "validity rate": validity_rate,
        "uniqueness rate": uniqueness_rate,
        "novelty rate": novelty_rate,
        "scaffold novelty rate": scaffold_novelty_rate,
        "n_generated": len(generated),
        "n_valid": len(valid_smiles),
    })

results_df = pd.DataFrame(results)
results_df.to_csv(results_dir / f"{model_name}_metrics.csv", index=False)
results_df

[01:35:19] SMILES Parse Error: extra open parentheses while parsing: O=c1cc(-c2ccccc2)c(-c2cccc(C(F)(F)F)c2)nc(SCC(=O)N[C@@H]2CCS(=O)(=O)
[01:35:19] SMILES Parse Error: check for mistakes around position 42:
[01:35:19] 2cccc(C(F)(F)F)c2)nc(SCC(=O)N[C@@H]2CCS(=
[01:35:19] ~~~~~~~~~~~~~~~~~~~~^
[01:35:19] SMILES Parse Error: Failed parsing SMILES 'O=c1cc(-c2ccccc2)c(-c2cccc(C(F)(F)F)c2)nc(SCC(=O)N[C@@H]2CCS(=O)(=O)' for input: 'O=c1cc(-c2ccccc2)c(-c2cccc(C(F)(F)F)c2)nc(SCC(=O)N[C@@H]2CCS(=O)(=O)'
[01:35:19] Can't kekulize mol.  Unkekulized atoms: 10 17 18
[01:35:19] Can't kekulize mol.  Unkekulized atoms: 10 17 18
[01:35:19] Can't kekulize mol.  Unkekulized atoms: 6 7 8 12 13 14
[01:35:19] Can't kekulize mol.  Unkekulized atoms: 6 7 8 12 13 14
[01:35:19] Can't kekulize mol.  Unkekulized atoms: 13 14 15 16 17 18 19
[01:35:19] Can't kekulize mol.  Unkekulized atoms: 13 14 15 16 17 18 19
[01:35:19] Can't kekulize mol.  Unkekulized atoms: 11 12 15
[01:35:19] Can't kekulize mol.  Unkekulized 

,temperature,validity rate,uniqueness rate,novelty rate,scaffold novelty rate,n_generated,n_valid
0,0.7,0.9830,1.0,0.995931,0.092065,2000,1966
1,1.0,0.9115,1.0,0.998354,0.073505,2000,1823
2,1.3,0.7915,1.0,1.000000,0.054959,2000,1583
